> **Pipeline status:** DEPRECATED for publication runs. Use 10_Event_Based_Early_Warning_LPPLS_POPRAWIONY.ipynb and 10b instead. This notebook is retained only for historical comparison.


# Notatnik 10 — Event-Based Early Warning Evaluation

**Cel:** ocena prognoz modeli z notatnika **08b** jako systemu ostrzegania przed spadkami BTC na poziomie **zdarzeń**, nie tylko pojedynczych dni.

Wyniki:

1. Wykres ceny BTC z **różowymi historycznymi bańkami UNION01–UNION14** i **szarymi okresami alarmowymi modelu** (format zbliżony do przykładu).
2. Wykres przybliżający zdarzenia spadkowe i wyprzedzające je alarmy w latach OOS.
3. Tabela **Event Recall, Alarm Precision, Median Lead Time, fałszywych alarmów/rok oraz Time in Alarm** dla modeli Price-only / LPPLS-only / Price+LPPLS.
4. Tabele konkretnych zdarzeń i alarmów do dalszej analizy.

**Co jest zdarzeniem?** W wersji podstawowej zdarzenie spadkowe to **pierwsze przekroczenie spadku ceny o co najmniej D% względem maksimum z minionych 180 dni**, po scaleniu przekroczeń rozdzielonych krótszym odstępem niż 60 dni. **Dzień zdarzenia = dzień przekroczenia progu**, a nie dzień szczytu, dno spadku ani początek bańki. To odrębna definicja od `drop_10_h60` (spadek od ceny w dniu prognozy), dlatego ocena event-based jest dodatkowym testem praktycznym, a nie zamiennikiem klasycznych metryk.

**Uwaga metodologiczna:** wszystkie alarmy są pobierane z zapisanych **prognoz OOS `y_pred`** z 08b. Nie dobieramy nowych progów na danych testowych. Wybór modelu, scenariusza i sposobu definicji epizodu był już jednak inspirowany wynikami 2021–2024, więc eksperyment ma charakter **retrospektywny / eksploracyjny**.

## Komórka 1 — biblioteki i ustawienia

Wymagane biblioteki: `pandas`, `numpy`, `matplotlib` oraz `scikit-learn` (ostatnia tylko dla dodatkowego AP). Notebook nie trenuje ponownie żadnego modelu.

In [ ]:
import sys as _sys
from pathlib import Path as _ResearchPath
_sys.path.insert(0,str(_ResearchPath('.').resolve()/'src'))
from lppls_research import TemporalAvailabilityValidator
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from IPython.display import display
from sklearn.metrics import average_precision_score

ROOT = Path('.')
OUT = ROOT / 'wyniki_event_based_10'
OUT.mkdir(parents=True, exist_ok=True)

PRED_CANDIDATES = [
    ROOT / 'wyniki_tuning_08b' / '08b_04_predykcje_OOS.csv',
    ROOT / '08b_04_predykcje_OOS.csv',
]
PRICE_CANDIDATES = [
    ROOT / 'wyniki_future_drop_07' / 'lppls_future_drop_dataset.csv',
    ROOT / 'lppls_future_drop_dataset.csv',
    ROOT / 'btc_usd_2015_2024.csv',
]

# (próg spadku %, horyzont dni) — dla wszystkich dziewięciu ustaw listę z komentarza
TARGET_CONFIGS = [(10, 60)]
# TARGET_CONFIGS = [(d,h) for h in (30, 45, 60) for d in (10, 15, 20)]

MODELS_TO_COMPARE = ['LogisticRegression', 'XGBoost', 'LightGBM']
FEATURE_GROUPS = ['Price-only', 'LPPLS-only', 'Price+LPPLS']
PLOT_MODEL = 'XGBoost'
PLOT_FEATURE_GROUP = 'Price+LPPLS'
PLOT_DROP, PLOT_HORIZON = 10, 60

# Z góry ustalone definicje, bez optymalizacji na okresie testowym
ROLLING_PEAK_DAYS = 180    # historyczne maksimum używane do definicji wydarzenia
MIN_PEAK_HISTORY_DAYS = 180
EVENT_MIN_GAP_DAYS = 60    # grupowanie kolejnych przekroczeń progu
ALARM_MERGE_GAP_DAYS = 7   # <=7 dni ciszy pomiędzy alarmami w jednym epizodzie
MIN_EVENT_LOOKBACK_DAYS = 14  # minimum dostępnej historii OOS przed zdarzeniem
LOG_PRICE_SCALE = False   # True, jeśli chcesz lepiej widzieć lata 2015–2018

# Historię UNION zachowujemy wyłącznie jako kontekst wizualny, NIE do predykcji.
DATA_UNION = [
    ('UNION01', '2016-06-08', '2016-06-18'),
    ('UNION02', '2016-12-30', '2017-01-15'),
    ('UNION03', '2017-02-22', '2017-03-07'),
    ('UNION04', '2017-04-27', '2017-07-05'),
    ('UNION05', '2017-07-15', '2018-01-09'),
    ('UNION06', '2019-06-17', '2019-07-13'),
    ('UNION07', '2019-08-09', '2019-08-10'),
    ('UNION08', '2019-12-06', '2020-01-05'),
    ('UNION09', '2020-02-07', '2020-02-17'),
    ('UNION10', '2020-03-20', '2020-09-21'),
    ('UNION11', '2020-11-06', '2021-05-20'),
    ('UNION12', '2021-08-08', '2021-09-19'),
    ('UNION13', '2021-10-05', '2021-12-02'),
    ('UNION14', '2024-03-04', '2024-03-25'),
]
union = pd.DataFrame(DATA_UNION, columns=['bubble_id','start','end'])
union['start'] = pd.to_datetime(union['start'])
union['end'] = pd.to_datetime(union['end'])

print('Scenariusze:', TARGET_CONFIGS)
print('Wyniki będą w:', OUT.resolve())

## Komórka 2 — wczytanie rzeczywistych prognoz 08b i cen BTC

Nie używaj symulowanych plików z testów technicznych. `08b_04_predykcje_OOS.csv` musi pochodzić z rzeczywistego uruchomienia 08b na BTC, a ceny z tego samego okresu. Jeśli notebook nie znajdzie pliku, zgłosi jasny komunikat.

In [ ]:
def locate(candidates, name):
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f'Brak pliku {name}. Szukano w: ' + ', '.join(map(str, candidates))
    )

pred_path = locate(PRED_CANDIDATES, '08b_04_predykcje_OOS.csv')
price_path = locate(PRICE_CANDIDATES, 'cen BTC')

preds = pd.read_csv(pred_path)
required = {'date', 'drop_pct', 'horizon', 'model', 'feature_group', 'y_true', 'y_pred', 'score'}
missing = required - set(preds.columns)
if missing:
    raise ValueError(f'Brakujące kolumny w prognozach OOS: {sorted(missing)}')
preds['date'] = pd.to_datetime(preds['date'], utc=True).dt.tz_convert(None).dt.normalize()
for c in ['drop_pct', 'horizon', 'y_true', 'y_pred']:
    preds[c] = pd.to_numeric(preds[c], errors='raise').astype(int)
preds['score'] = pd.to_numeric(preds['score'], errors='raise')
assert preds[['y_true', 'y_pred']].isin([0,1]).all().all()
assert preds['score'].between(0,1).all()
assert not preds.duplicated(['date', 'drop_pct', 'horizon', 'model', 'feature_group']).any(), 'Duplikaty dziennych prognoz OOS'

prices = pd.read_csv(price_path)
if 'Date' in prices.columns and 'date' not in prices.columns:
    prices = prices.rename(columns={'Date': 'date'})
if 'Close' in prices.columns and 'price' not in prices.columns:
    prices = prices.rename(columns={'Close': 'price'})
if not {'date','price'}.issubset(prices.columns):
    raise ValueError('Plik cen powinien zawierać date + price (lub Date + Close)')
prices = prices[['date', 'price']].copy()
prices['date'] = pd.to_datetime(prices['date'], utc=True).dt.tz_convert(None).dt.normalize()
prices['price'] = pd.to_numeric(prices['price'], errors='raise')
prices = prices.sort_values('date').drop_duplicates('date').reset_index(drop=True)
assert prices['price'].gt(0).all() and prices['price'].notna().all()
assert prices['date'].diff().dropna().eq(pd.Timedelta(days=1)).all(), 'Brak dat w szeregu cen BTC'

if not preds['date'].isin(prices['date']).all():
    raise ValueError('Nie wszystkie dni z prognoz 08b znajdują się w cenach BTC')

print('Prognozy:', pred_path, '| wierszy:', len(preds))
print('Ceny:', price_path, '| dni:', len(prices))
print('Rzeczywiste zakresy:', prices.date.min().date(), '—', prices.date.max().date())
print('Dostępne kombinacje modeli:')
display(preds[['drop_pct', 'horizon', 'model', 'feature_group']].drop_duplicates().head(25))

## Komórka 3 — niezależna identyfikacja zdarzeń spadkowych

Rozpoznajemy **pierwszy dzień przekroczenia** progu względem historycznego maksimum z 180 dni. Kolejne przekroczenia zbyt bliskie czasowo łączymy, żeby jedna większa przecena nie była liczona wielokrotnie.

Ta definicja jest świadomie **inna** niż etykieta z notatnika 07: tam liczyliśmy spadek w ciągu przyszłych H dni **od ceny w dniu prognozy**, a tutaj zdarzenie oznacza wejście w stan drawdown względem wcześniejszego maksimum. Nie wolno interpretować ich jako identycznych celów.

In [ ]:
def detect_drawdown_events(price_df, drop_pct, rolling_peak_days=180,
                           min_history_days=180, min_gap_days=60):
    p = price_df.sort_values('date').reset_index(drop=True).copy()
    rolling_peak = p['price'].rolling(
        rolling_peak_days, min_periods=min_history_days
    ).max()
    p['reference_peak'] = rolling_peak
    p['drawdown'] = p['price'] / rolling_peak - 1.0
    below = p['drawdown'].le(-drop_pct / 100.0).fillna(False)
    crossing = below & ~below.shift(1, fill_value=False)

    results = []
    last_event_date = None
    for row in p.loc[crossing].itertuples(index=False):
        when = row.date
        if last_event_date is not None and (when - last_event_date).days < min_gap_days:
            continue
        results.append({
            'event_id': f'DROP{drop_pct}_{len(results)+1:03d}',
            'event_date': when,
            'drop_pct': int(drop_pct),
            'price_at_crossing': float(row.price),
            'reference_peak': float(row.reference_peak),
            'drawdown_at_crossing': float(row.drawdown),
        })
        last_event_date = when
    return pd.DataFrame(results, columns=[
        'event_id','event_date','drop_pct','price_at_crossing',
        'reference_peak','drawdown_at_crossing'
    ])

price_events = {
    D: detect_drawdown_events(
        prices, D, ROLLING_PEAK_DAYS,
        MIN_PEAK_HISTORY_DAYS, EVENT_MIN_GAP_DAYS
    )
    for D in sorted({D for D, H in TARGET_CONFIGS})
}

for D, frame in price_events.items():
    print(f'Przekroczenia drawdown {D}% w całej historii:', len(frame))
    display(frame.tail(8))

## Komórka 4 — alarmy modeli i agregacja w epizody

Jeśli model generuje ostrzeżenie dzień po dniu, nie liczymy każdego dnia jako osobnego alarmu. Scalamy dni alarmowe oddzielone **co najwyżej 7 dniami bez alarmu**; początek epizodu alarmowego to data pierwszego ostrzeżenia w danym klastrze.

In [ ]:
def get_daily_predictions(preds, drop, horizon, model, feature_group):
    part = preds.loc[
        preds['drop_pct'].eq(drop)
        & preds['horizon'].eq(horizon)
        & preds['model'].eq(model)
        & preds['feature_group'].eq(feature_group)
    ].copy().sort_values('date').reset_index(drop=True)
    if part.empty:
        return part
    assert part['date'].is_unique
    return part


def make_alarm_episodes(daily_pred, gap_days=7):
    dates = list(daily_pred.loc[daily_pred['y_pred'].eq(1), 'date'].sort_values())
    rows = []
    if not dates:
        return pd.DataFrame(columns=['alarm_id','start','end','alarm_days'])
    start = end = dates[0]
    days = 1
    for date in dates[1:]:
        if (date - end).days <= gap_days + 1:
            end = date
            days += 1
        else:
            rows.append((start, end, days))
            start = end = date
            days = 1
    rows.append((start, end, days))
    return pd.DataFrame([
        {'alarm_id': f'ALARM_{i+1:03d}', 'start': a, 'end': b, 'alarm_days': n}
        for i, (a,b,n) in enumerate(rows)
    ])

print('Przykładowe grupowanie alarmów:')
example = get_daily_predictions(preds, PLOT_DROP, PLOT_HORIZON, PLOT_MODEL, PLOT_FEATURE_GROUP)
if example.empty:
    warnings.warn('Nie znaleziono ustawionej konfiguracji wykresu; wybierz inną w Komórce 1.')
else:
    display(make_alarm_episodes(example, ALARM_MERGE_GAP_DAYS).head(10))

## Komórka 5 — dopasowanie alarmów do przyszłych zdarzeń

**Wersja konserwatywna:**
- Alarm jest poprawny, jeśli jego **początek** pojawia się **od 1 do H dni przed dniem przekroczenia progu spadku**.
- Każdy alarm można przyporządkować do **najwyżej jednego** zdarzenia, a każde zdarzenie może zostać wykryte najwyżej raz.
- Alarmy z niepełnym H-dniowym okresem obserwacji prognoz OOS nie są oceniane jako fałszywe (ich wynik jest niedostępny).
- Zdarzenie jest oceniane tylko wtedy, gdy istnieje wystarczający odcinek prognoz OOS przed jego wystąpieniem (minimum 14 dni i bez braków w tym odcinku).
- Liczba fałszywych alarmów oznacza **niepowiązane epizody alarmowe**, nie liczbę dni.

Wybór tych reguł ma wpływ na wyniki — do publikacji należy sprawdzić ich wrażliwość (np. odstęp 3/7/14 dni i horyzont ostrzegania 30/45/60 dni).

In [ ]:
def event_based_evaluation(daily_pred, all_events, horizon,
                           merge_gap_days=7, min_event_lookback_days=14):
    d = daily_pred.sort_values('date').reset_index(drop=True).copy()
    if d.empty:
        return {}, pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    date_set = set(d['date'])
    min_date, max_date = d['date'].min(), d['date'].max()

    # Czy można w całości ocenić H przyszłych dni w ramach dostępnego OOS?
    # Wykluczamy końcówkę OOS, aby nie zaniżać liczby prawdziwych zdarzeń.
    evaluable_dates = set()
    for date in d['date']:
        if all(date + pd.Timedelta(days=i) in date_set for i in range(1, horizon+1)):
            evaluable_dates.add(date)

    alarms = make_alarm_episodes(d, merge_gap_days)
    alarms['eligible'] = alarms['start'].isin(evaluable_dates) if len(alarms) else pd.Series(dtype=bool)
    alarms['matched_event_id'] = ''
    alarms['false_alarm'] = False

    # Zdarzenia, przed którymi istnieje rzeczywisty ciąg prognoz OOS
    events = all_events.loc[
        all_events['event_date'].between(min_date, max_date)
    ].copy().sort_values('event_date').reset_index(drop=True)
    event_eligible = []
    for day in events['event_date']:
        available = min(horizon, (day - min_date).days)
        if available < min_event_lookback_days:
            event_eligible.append(False)
            continue
        event_eligible.append(all(
            day - pd.Timedelta(days=i) in date_set
            for i in range(1, available + 1)
        ))
    events['eligible'] = event_eligible
    events['detected'] = False
    events['matched_alarm_id'] = ''
    events['alarm_start'] = pd.NaT
    events['lead_time_days'] = np.nan

    used_alarms = set()
    for idx, event in events.loc[events['eligible']].iterrows():
        lower = event['event_date'] - pd.Timedelta(days=horizon)
        upper = event['event_date'] - pd.Timedelta(days=1)
        candidates = alarms.loc[
            alarms['eligible']
            & alarms['start'].between(lower, upper)
            & ~alarms['alarm_id'].isin(used_alarms)
        ].sort_values('start')
        if candidates.empty:
            continue
        alarm_idx = candidates.index[0]
        alarm = candidates.iloc[0]
        used_alarms.add(alarm['alarm_id'])
        events.loc[idx, 'detected'] = True
        events.loc[idx, 'matched_alarm_id'] = alarm['alarm_id']
        events.loc[idx, 'alarm_start'] = alarm['start']
        events.loc[idx, 'lead_time_days'] = (event['event_date']-alarm['start']).days
        alarms.loc[alarm_idx, 'matched_event_id'] = event['event_id']

    if len(alarms):
        alarms['false_alarm'] = alarms['eligible'] & alarms['matched_event_id'].eq('')

    n_events = int(events['eligible'].sum())
    n_detected = int(events.loc[events['eligible'], 'detected'].sum())
    n_alarms = int(alarms['eligible'].sum())
    n_false_alarms = int(alarms['false_alarm'].sum())
    n_matched_alarms = n_alarms - n_false_alarms
    leads = events.loc[events['detected'], 'lead_time_days']

    # Czas, w którym faktycznie dało się przeprowadzić kompletną ocenę H dni
    valid_mask = d['date'].isin(evaluable_dates)
    eval_days = int(valid_mask.sum())
    metrics = {
        'prediction_days': int(len(d)),
        'evaluated_days': eval_days,
        'n_events': n_events,
        'events_detected': n_detected,
        'event_recall': n_detected/n_events if n_events else np.nan,
        'alarm_episodes': n_alarms,
        'matched_alarm_episodes': n_matched_alarms,
        'false_alarm_episodes': n_false_alarms,
        'alarm_precision': n_matched_alarms/n_alarms if n_alarms else np.nan,
        'false_alarms_per_year': n_false_alarms/(eval_days/365.25) if eval_days else np.nan,
        'median_lead_days': float(leads.median()) if not leads.empty else np.nan,
        'mean_lead_days': float(leads.mean()) if not leads.empty else np.nan,
        'time_in_alarm': float(d.loc[valid_mask, 'y_pred'].mean()) if eval_days else np.nan,
    }
    return metrics, events, alarms, d.loc[valid_mask].copy()

## Komórka 6 — ocena porównawcza modeli i zapis wyników

W tabelach jest osobny wiersz dla **każdego modelu i grupy cech**. Wyniki event-based nie muszą być w pełni porównywalne z metrykami z 08b, bo używają innej definicji zdarzenia i nie zawsze oceniają dokładnie te same dni.

In [ ]:
summary_rows = []
event_rows = []
alarm_rows = []
run_cache = {}

for D, H in TARGET_CONFIGS:
    source_events = price_events.get(D)
    if source_events is None:
        source_events = detect_drawdown_events(
            prices, D, ROLLING_PEAK_DAYS,
            MIN_PEAK_HISTORY_DAYS, EVENT_MIN_GAP_DAYS
        )
    for model in MODELS_TO_COMPARE:
        for feature_group in FEATURE_GROUPS:
            d = get_daily_predictions(preds, D, H, model, feature_group)
            if d.empty:
                print(f'Pomijam niedostępny wariant: {D}%/{H}d — {model} / {feature_group}')
                continue
            m, ev, al, valid_days = event_based_evaluation(
                d, source_events, H,
                ALARM_MERGE_GAP_DAYS, MIN_EVENT_LOOKBACK_DAYS
            )
            key = (D, H, model, feature_group)
            run_cache[key] = dict(daily=d, events=ev, alarms=al, valid_days=valid_days)
            ap = (average_precision_score(d['y_true'], d['score'])
                  if d['y_true'].nunique() > 1 else np.nan)
            summary_rows.append({
                'drop_pct': D, 'horizon': H,
                'model': model, 'feature_group': feature_group,
                'daily_positive_rate': float(d['y_true'].mean()),
                'daily_AP': float(ap),
                **m,
            })
            for _, row in ev.iterrows():
                event_rows.append({
                    'drop_pct': D, 'horizon': H,
                    'model': model, 'feature_group': feature_group,
                    **row.to_dict(),
                })
            for _, row in al.iterrows():
                alarm_rows.append({
                    'drop_pct': D, 'horizon': H,
                    'model': model, 'feature_group': feature_group,
                    **row.to_dict(),
                })

summary = pd.DataFrame(summary_rows)
if summary.empty:
    raise RuntimeError('Brak dostępnych prognoz OOS z wybranych modeli/scenariuszy. Zmień konfigurację w komórce 1.')
all_events_results = pd.DataFrame(event_rows)
all_alarm_results = pd.DataFrame(alarm_rows)

summary.to_csv(OUT/'10_01_porownanie_event_based.csv', index=False)
all_events_results.to_csv(OUT/'10_02_szczegoly_zdarzen.csv', index=False)
all_alarm_results.to_csv(OUT/'10_03_epizody_alarmowe.csv', index=False)
print('Porównanie event-based:')
display(summary.round(3))

## Komórka 7 — **wykres taki jak w przykładzie użytkownika**

Wykres pełnej historii 2015–2024:

- **czarna linia:** cena BTC;
- **różowe pasma:** historyczne okresy baniek UNION z literatury;
- **szare pasma:** alarmy wybranego modelu na podstawie jego prognoz OOS (08b);
- **granica OOS:** pionowa kropkowana linia, od której w ogóle mamy prognozy testowe.

Szare pasma NIE są „potwierdzonymi bańkami”, tylko okresami, w których model ostrzegał. Nie zamieniaj ich nazwą na `Detected bubbles` bez osobnej procedury weryfikacji.

In [ ]:
def plot_price_bubbles_and_alarms(price_df, union_df, alarm_episodes, oos_start,
                                  title, destination, xlim=None, log_scale=False):
    fig, ax = plt.subplots(figsize=(16, 6.6))
    ax.plot(price_df['date'], price_df['price'], color='#222222', lw=1.25, label='Cena BTC')

    for row in union_df.itertuples(index=False):
        ax.axvspan(row.start, row.end + pd.Timedelta(days=1),
                   color='#e66b71', alpha=0.23, linewidth=0)
    for row in alarm_episodes.itertuples(index=False):
        ax.axvspan(row.start, row.end + pd.Timedelta(days=1),
                   color='#777777', alpha=0.27, linewidth=0)

    ax.axvline(oos_start, linestyle=':', color='#376a9a', lw=1.5)
    ax.set_title(title)
    ax.set_ylabel('BTC-USD — cena [USD]')
    ax.set_xlabel('Data')
    ax.grid(True, alpha=0.16)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    if xlim:
        ax.set_xlim(*xlim)
    else:
        ax.set_xlim(price_df['date'].min(), price_df['date'].max())
    if log_scale:
        ax.set_yscale('log')
    ax.legend(handles=[
        Line2D([0],[0], color='#222222', lw=1.4, label='Cena BTC'),
        Patch(color='#e66b71', alpha=0.3, label='Historyczne bańki (UNION)'),
        Patch(color='#777777', alpha=0.35, label='Alarmy modelu OOS'),
        Line2D([0],[0], color='#376a9a', linestyle=':', label='Początek OOS'),
    ], loc='upper left', fontsize=9)
    fig.tight_layout()
    fig.savefig(destination, dpi=200)
    plt.show()

plot_key = (PLOT_DROP, PLOT_HORIZON, PLOT_MODEL, PLOT_FEATURE_GROUP)
if plot_key not in run_cache:
    plot_key = next(iter(run_cache))
    warnings.warn(f'Wybrana konfiguracja nie była dostępna. Rysuję {plot_key}.')

D, H, model, group = plot_key
res = run_cache[plot_key]
plot_price_bubbles_and_alarms(
    prices, union, res['alarms'], res['daily']['date'].min(),
    f'Bitcoin — historyczne bańki i alarmy OOS ({model}, {group})',
    OUT/'10_04_cena_btc_banki_alarmy_2015_2024.png',
    log_scale=LOG_PRICE_SCALE,
)

## Komórka 8 — wykres z naciskiem na spadki i wyprzedzenie ostrzeżeń

Ten wykres powiększa tylko okres prognoz OOS. Oprócz **szarych alarmów** pokazuje **czerwone znaczniki pierwszego przekroczenia progu drawdown** (zdarzeń spadkowych), z którymi porównujemy ostrzeżenia.

In [ ]:
def plot_event_alert_timeline(price_df, events, alarms, first_date, last_date,
                              title, destination):
    fig, ax = plt.subplots(figsize=(15, 6))
    in_view = price_df['date'].between(first_date, last_date)
    pp = price_df.loc[in_view]
    ax.plot(pp['date'], pp['price'], color='#252525', lw=1.3)
    for row in alarms.itertuples(index=False):
        if row.end >= first_date and row.start <= last_date:
            ax.axvspan(row.start, row.end + pd.Timedelta(days=1),
                       color='#808080', alpha=.29, linewidth=0)
    matched = events.loc[events['eligible']]
    marked = matched['event_date'].between(first_date, last_date)
    for row in matched.loc[marked].itertuples(index=False):
        ax.axvline(row.event_date, color='#ae4046', alpha=.5, linestyle='--', lw=1.2)
    ax.scatter(matched.loc[marked, 'event_date'], matched.loc[marked, 'price_at_crossing'],
               color='#ae4046', marker='v', s=55, zorder=5)
    ax.set_xlim(first_date, last_date)
    ax.set_title(title)
    ax.set_ylabel('BTC-USD — cena [USD]')
    ax.set_xlabel('Data (OOS)')
    ax.grid(alpha=.18)
    ax.legend(handles=[
        Line2D([0],[0], color='#252525', lw=1.3, label='Cena BTC'),
        Patch(color='#808080', alpha=.35, label='Okresy alarmowe'),
        Line2D([0],[0], marker='v', linestyle='--', color='#ae4046',
               label='Przekroczenie progu spadku', markersize=7),
    ], loc='upper left')
    fig.tight_layout()
    fig.savefig(destination, dpi=200)
    plt.show()

plot_event_alert_timeline(
    prices, res['events'], res['alarms'],
    res['daily']['date'].min(), res['daily']['date'].max(),
    f'Wczesne ostrzeganie OOS — spadek ≥{D}% / {H} dni — {model} ({group})',
    OUT/'10_05_cena_btc_zdarzenia_alarmy_OOS.png'
)

print('Zdarzenia ocenialne i dopasowane:')
display(res['events'][['event_id','event_date','eligible','detected',
                       'matched_alarm_id','lead_time_days']].round(2))

## Komórka 9 — dodatkowe porównanie modeli na wykresie

Wykres `Event Recall` versus `False Alarms per Year` pozwala zobaczyć praktyczny kompromis. Modele o podobnym AP mogą być bardzo różne z perspektywy liczby niepotrzebnych ostrzeżeń.

In [ ]:
for D, H in TARGET_CONFIGS:
    part = summary.loc[summary.drop_pct.eq(D) & summary.horizon.eq(H)].copy()
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for group, subset in part.groupby('feature_group', sort=False):
        ax.scatter(subset['false_alarms_per_year'], subset['event_recall'],
                   s=75, label=group)
        for row in subset.itertuples(index=False):
            ax.annotate(row.model, (row.false_alarms_per_year, row.event_recall),
                        xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax.set_xlabel('Fałszywe epizody alarmowe / rok')
    ax.set_ylabel('Event Recall')
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=.2)
    ax.set_title(f'Wczesne ostrzeganie: czułość vs fałszywe alarmy — {D}%/{H}d')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT/f'10_06_event_recall_fa_{D}_{H}.png', dpi=180)
    plt.show()

## Komórka 10 — interpretacja i ograniczenia

**Odczyt wyników do publikacji:**

- `event_recall` — odsetek ocenialnych zdarzeń, przed którymi był co najmniej jeden niezależny alarm;
- `median_lead_days` — mediana odległości w dniach między początkiem alarmu a pierwszym przekroczeniem progu spadku;
- `false_alarm_episodes` — liczba ostrzeżeń niepowiązanych ze zdarzeniem w zdefiniowanym horyzoncie;
- `false_alarms_per_year` — ta liczba przeskalowana przez czas ocenialnych dni OOS;
- `time_in_alarm` — udział dni w stanie alarmowym;
- `daily_AP` — oryginalna metryka jakości rankingu z notatnika 08b, pokazana dla porównania; nie należy porównywać jej liczbowo z event recall.

### Niezbędne zastrzeżenia

1. **Prawdziwe zdarzenia** wyznaczamy automatycznie z ceny (próg drawdown), a **historyczne bańki** z literatury pokazujemy tylko jako kontekst. To dwa różne pojęcia.
2. Definicja zdarzenia nie odpowiada dokładnie etykiecie `drop_D_hH` z notatnika 07. Wyniki event-based są dodatkową oceną zastosowania, nie nową metryką dla identycznego celu.
3. Reguły scalania alarmów (7 dni), odstęp zdarzeń (60 dni), historyczne maksimum (180 dni) i minimalny okres danych (14 dni) muszą mieć **analizę wrażliwości**. Nie należy dopasowywać ich do testu, aby poprawić wynik.
4. Nie wyciągaj wniosków o formalnej istotności statystycznej z samego event recall, szczególnie gdy liczba wydarzeń jest mała.
5. Alarm po wystąpieniu przekroczenia progu nie jest zaliczony jako wczesne ostrzeżenie.
6. Szare pasma = **alarmy OOS**, a nie rozpoznane po fakcie „detected bubbles”. Historyczne okresy UNION nie są wprowadzane do modelu.
7. Ponieważ 08b był już analizowany na latach 2021–2024, wyniki dotyczą okresu **wielokrotnie eksplorowanego**, a nie całkowicie nietkniętego testu.

Po uruchomieniu otrzymasz tabelę `10_01_porownanie_event_based.csv`, szczegóły dopasowania `10_02_szczegoly_zdarzen.csv` i epizody alarmowe `10_03_epizody_alarmowe.csv` oraz wykresy PNG.